# EDA v1 - RentSense

Анализ данных о предложениях аренды недвижимости в Москве.

**Цель:** понять структуру данных, выявить проблемы и подготовиться к обучению моделей.

**Время и валидация:** для объёма выборки и time-based split ориентируемся на **`offers.created_at`** (в выборке — `offer_created_at`): это дата попадания объявления в нашу БД. При **короткой истории** (несколько месяцев) не имеет смысла дополнительно сужать train «по свежести»; разумнее **отложить последний месяц или 1–2 недели** в test и оценивать качество на «будущем» относительно train. Разбиение только по `publication_at` «последние 30 дней от максимума» при длинном хвосте старых публикаций даёт **перекос** долей train/test (см. раздел 6).


In [1]:
# Импорты библиотек
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import dotenv_values
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Настройка визуализации
try:
    plt.style.use('seaborn-v0_8')
except:
    try:
        plt.style.use('seaborn')
    except:
        plt.style.use('default')

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Библиотеки загружены")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}, Matplotlib: {matplotlib.__version__}")


Библиотеки загружены
NumPy: 1.26.4, Pandas: 2.1.4, Matplotlib: 3.8.2


## 1. Подключение к БД и загрузка данных


In [2]:
# Подключение к БД (удаленный сервер 89.110.92.128)
from pathlib import Path
import sys
import subprocess

# Проверка и установка pymysql если нужно
try:
    import pymysql
except ImportError:
    print("Устанавливаю pymysql...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymysql", "--quiet"])
    import pymysql

env_path = Path('../..') / '.env'
env = dotenv_values(env_path)

DBTYPE = env.get('DB_TYPE') or 'mysql+pymysql'
LOGIN = env.get('DB_LOGIN') or 'root'
PASS = env.get('DB_PASS') or 'rootpassword'
IP = env.get('DB_IP') or '89.110.92.128'
PORT = env.get('DB_PORT') or '3306'
DBNAME = env.get('DB_NAME') or 'rentsense'

# Показываем, какие параметры используются (без пароля)
print(f"Параметры подключения:")
print(f"  IP: {IP}")
print(f"  PORT: {PORT}")
print(f"  USER: {LOGIN}")
print(f"  DB: {DBNAME}")

DATABASE_URL = f'{DBTYPE}://{LOGIN}:{PASS}@{IP}:{PORT}/{DBNAME}?charset=utf8mb4'

print(f"Подключение к БД: {DBNAME}@{IP}:{PORT}")
print(f"Параметры из .env: IP={IP}, PORT={PORT}")

try:
    engine = create_engine(DATABASE_URL, pool_pre_ping=True, connect_args={"connect_timeout": 10})
    # Проверка подключения
    with engine.connect() as conn:
        print("Подключение успешно")
except Exception as e:
    print(f"Ошибка подключения: {e}")
    print("\nПроверьте:")
    print("1. Файл .env с параметрами DB_IP, DB_PORT, DB_LOGIN, DB_PASS")
    print("2. Доступность сервера БД (89.110.92.128:3306)")
    print("3. Правильность учетных данных")
    print("4. Firewall/сеть позволяет подключение к удаленному серверу")
    raise


Параметры подключения:
  IP: 89.110.92.128
  PORT: 3306
  USER: rentsense
  DB: rentsense
Подключение к БД: rentsense@89.110.92.128:3306
Параметры из .env: IP=89.110.92.128, PORT=3306


Подключение успешно


In [3]:
# Загрузка данных
# Проверка, что engine определен (ячейка 3 должна выполниться успешно)
if 'engine' not in locals() and 'engine' not in globals():
    raise NameError("Переменная 'engine' не определена. Сначала запустите ячейку 3 (подключение к БД)")

query = """
SELECT 
    o.cian_id,
    o.price,
    o.price_changes,
    o.category,
    o.views_count,
    o.photos_count,
    o.floor_number,
    o.floors_count,
    o.publication_at,
    o.created_at as offer_created_at,
    o.updated_at as offer_updated_at,
    
    a.county,
    a.district,
    a.street,
    a.house,
    a.metro,
    a.travel_type,
    a.travel_time,
    a.coordinates,
    
    ri.repair_type,
    ri.total_area,
    ri.living_area,
    ri.kitchen_area,
    ri.ceiling_height,
    ri.balconies,
    ri.loggias,
    ri.rooms_count,
    ri.separated_wc,
    ri.combined_wc,
    ri.windows_view,
    
    ro.build_year,
    ro.entrances,
    ro.material_type,
    ro.parking_type,
    ro.garbage_chute,
    ro.lifts_count,
    ro.passenger_lifts,
    ro.cargo_lifts,
    
    rd.realty_type,
    rd.project_type,
    rd.heat_type,
    rd.gas_type,
    rd.is_apartment,
    rd.is_penthouse,
    rd.is_mortgage_allowed,
    rd.is_premium,
    rd.is_emergency,
    
    od.deal_type,
    od.flat_type,
    od.payment_period,
    od.deposit,
    od.prepay_months,
    od.utilities_included,
    od.client_fee,
    od.agent_fee,
    od.description,
    
    d.name as developer_name,
    d.review_count as developer_review_count,
    d.total_rate as developer_rate,
    d.buildings_count as developer_buildings_count,
    d.foundation_year as developer_foundation_year,
    d.is_reliable as developer_is_reliable
FROM offers o
LEFT JOIN addresses a ON o.cian_id = a.cian_id
LEFT JOIN realty_inside ri ON o.cian_id = ri.cian_id
LEFT JOIN realty_outside ro ON o.cian_id = ro.cian_id
LEFT JOIN realty_details rd ON o.cian_id = rd.cian_id
LEFT JOIN offers_details od ON o.cian_id = od.cian_id
LEFT JOIN developers d ON o.cian_id = d.cian_id
WHERE o.category = 'flatRent'
  AND (od.deal_type IS NULL OR od.deal_type = 'rent')
"""

print("Загрузка данных из БД (только flatRent + rent)...")
df = pd.read_sql(query, engine)
print(f"Загружено строк: {len(df)}")

if 'category' in df.columns:
    print(f"\nРаспределение по category:")
    print(df['category'].value_counts())

df.head()


Загрузка данных из БД (только flatRent + rent)...


Загружено строк: 46779

Распределение по category:
category
flatRent    46779
Name: count, dtype: int64


,cian_id,price,price_changes,category,views_count,photos_count,floor_number,floors_count,publication_at,offer_created_at,...,utilities_included,client_fee,agent_fee,description,developer_name,developer_review_count,developer_rate,developer_buildings_count,developer_foundation_year,developer_is_reliable
0,325573108,122400.0,"[{""priceData"": {""price"": 122400, ""currency"": ""...",flatRent,None,22.0,3,28,1.767697e+09,2026-01-09 15:35:40,...,1,0,0,Идут показы с датой заселения после 21.01.2026...,Галс-Девелопмент,194.0,4.7,178.0,1994.0,0.0
1,325353038,1200000.0,"[{""priceData"": {""price"": 1200000, ""currency"": ...",flatRent,None,48.0,12,18,1.766627e+09,2026-01-09 15:35:54,...,1,0,0,Лот 554223. Бонус коллегам! Предлагается в аре...,None,NaN,NaN,NaN,NaN,NaN
2,325047081,225000.0,"[{""priceData"": {""price"": 225000, ""currency"": ""...",flatRent,None,11.0,3,5,1.765795e+09,2026-01-09 15:36:41,...,1,0,0,Просторная квартира с 3 спальнями площадью 100...,None,NaN,NaN,NaN,NaN,NaN
3,325322890,1200000.0,"[{""priceData"": {""price"": 1200000, ""currency"": ...",flatRent,None,25.0,12,18,1.766530e+09,2026-01-09 15:46:19,...,1,0,0,Просторная квартира с 4 спальнями общей площад...,None,NaN,NaN,NaN,NaN,NaN
4,325225143,55000.0,"[{""priceData"": {""price"": 55000, ""currency"": ""r...",flatRent,None,20.0,3,5,1.766241e+09,2026-01-09 15:50:22,...,1,60,50,Сдаётся просторная 2-комнатная квартира с евро...,None,NaN,NaN,NaN,NaN,NaN


## 1.1 Горизонт данных по `created_at`

Для объёма выборки и time-based split важна **дата парсинга** (`offers.created_at` → в датафрейме `offer_created_at`). По ней видно, сколько объявлений попало в БД по месяцам. Публикация на ЦИАН (`publication_at`) может быть сильно раньше — см. раздел 6.

In [4]:
df['offer_created_at'] = pd.to_datetime(df['offer_created_at'], errors='coerce')
df['_ym_created'] = df['offer_created_at'].dt.to_period('M').astype(str)
month_created = df['_ym_created'].value_counts().sort_index()
print("Строк по месяцам (offer_created_at):")
print(month_created.to_string())
print(f"\nПропусков offer_created_at: {df['offer_created_at'].isna().sum()}")

fig, ax = plt.subplots(figsize=(12, 4))
month_created.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Количество объявлений по месяцу created_at (парсинг в БД)')
ax.set_xlabel('Месяц')
ax.set_ylabel('Количество')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Строк по месяцам (offer_created_at):
_ym_created
2026-01    10364
2026-02    19893
2026-03    16522

Пропусков offer_created_at: 0


### 1.2 Анализ price_changes и актуальной цены

Проверяем, актуальная ли цена в `offers.price` или нужно брать из `price_changes`


In [5]:
# Анализ price_changes - извлечение актуальной цены
import json

def get_latest_price_from_changes(price_changes_json):
    """Извлекает последнюю цену из price_changes по дате changeTime"""
    if not price_changes_json:
        return None
    try:
        if isinstance(price_changes_json, str):
            changes = json.loads(price_changes_json)
        else:
            changes = price_changes_json
            
        if not isinstance(changes, list) or len(changes) == 0:
            return None
        
        # Сортируем по changeTime (последняя дата = актуальная цена)
        sorted_changes = sorted(
            changes, 
            key=lambda x: x.get('changeTime', ''),
            reverse=True
        )
        
        latest = sorted_changes[0]
        return latest.get('priceData', {}).get('price')
    except Exception as e:
        return None

# Извлечение актуальной цены из price_changes
if 'price_changes' not in df.columns:
    print("price_changes не найдена, используем offers.price")
    df['price_from_changes'] = None
else:
    df['price_from_changes'] = df['price_changes'].apply(get_latest_price_from_changes)

# Сравнение цен
df['price_diff'] = df['price'] - pd.to_numeric(df['price_from_changes'], errors='coerce')
df['price_matches'] = (abs(df['price_diff']) < 0.01) | (df['price_from_changes'].isna())

if 'price_changes' in df.columns:
    print(f"Записей с price_changes: {df['price_changes'].notna().sum()} ({df['price_changes'].notna().sum()/len(df)*100:.1f}%)")
    print(f"Записей с актуальной ценой из price_changes: {df['price_from_changes'].notna().sum()} ({df['price_from_changes'].notna().sum()/len(df)*100:.1f}%)")
    print(f"\nСравнение цен:")
    print(f"  Цены совпадают: {df['price_matches'].sum()} ({df['price_matches'].sum()/len(df)*100:.1f}%)")
    print(f"  Цены различаются: {(~df['price_matches']).sum()} ({(~df['price_matches']).sum()/len(df)*100:.1f}%)")
    
    if (~df['price_matches']).sum() > 0:
        print(f"\nПримеры различий (первые 5):")
        diff_examples = df[~df['price_matches']][['cian_id', 'price', 'price_from_changes', 'price_diff']].head()
        print(diff_examples.to_string())
    
    df['price_actual'] = df['price_from_changes'].fillna(df['price'])
    print(f"\nАктуальная цена:")
    print(f"  Из price_changes: {(df['price_actual'] == df['price_from_changes']).sum()}")
    print(f"  Из offers.price: {(df['price_actual'] == df['price']).sum()}")
else:
    df['price_actual'] = df['price']


Записей с price_changes: 46522 (99.5%)
Записей с актуальной ценой из price_changes: 46522 (99.5%)

Сравнение цен:
  Цены совпадают: 46693 (99.8%)
  Цены различаются: 86 (0.2%)

Примеры различий (первые 5):
        cian_id     price  price_from_changes  price_diff
1539  325401597   90000.0             80000.0     10000.0
5132  326185036  109000.0            105000.0      4000.0
5133  326190710  110000.0            230000.0   -120000.0
5216  308500011   85000.0             90000.0     -5000.0
5246  324298426  100000.0            115000.0    -15000.0

Актуальная цена:
  Из price_changes: 46522
  Из offers.price: 46693


## 2. Общий обзор датасета


In [6]:
print(f"Размер датасета: {df.shape}")
print(f"\nКолонки ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nТипы данных:")
print(df.dtypes)
print(f"\nОсновная информация:")
df.info()


Размер датасета: (46779, 67)

Колонки (67):
['cian_id', 'price', 'price_changes', 'category', 'views_count', 'photos_count', 'floor_number', 'floors_count', 'publication_at', 'offer_created_at', 'offer_updated_at', 'county', 'district', 'street', 'house', 'metro', 'travel_type', 'travel_time', 'coordinates', 'repair_type', 'total_area', 'living_area', 'kitchen_area', 'ceiling_height', 'balconies', 'loggias', 'rooms_count', 'separated_wc', 'combined_wc', 'windows_view', 'build_year', 'entrances', 'material_type', 'parking_type', 'garbage_chute', 'lifts_count', 'passenger_lifts', 'cargo_lifts', 'realty_type', 'project_type', 'heat_type', 'gas_type', 'is_apartment', 'is_penthouse', 'is_mortgage_allowed', 'is_premium', 'is_emergency', 'deal_type', 'flat_type', 'payment_period', 'deposit', 'prepay_months', 'utilities_included', 'client_fee', 'agent_fee', 'description', 'developer_name', 'developer_review_count', 'developer_rate', 'developer_buildings_count', 'developer_foundation_year', 'de

In [7]:
# Статистика по числовым признакам
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("Описательная статистика числовых признаков:")
df[numeric_cols].describe()


Описательная статистика числовых признаков:


,cian_id,price,photos_count,floor_number,floors_count,publication_at,travel_time,total_area,living_area,kitchen_area,...,client_fee,agent_fee,developer_review_count,developer_rate,developer_buildings_count,developer_foundation_year,developer_is_reliable,price_from_changes,price_diff,price_actual
count,4.677900e+04,4.677900e+04,46583.000000,46779.000000,46779.000000,4.676700e+04,46519.000000,46779.000000,35651.000000,38793.000000,...,46779.000000,46779.000000,19232.000000,19232.000000,19480.000000,19230.000000,19690.0,4.652200e+04,46522.000000,4.677900e+04
mean,3.228134e+08,1.482789e+05,16.611081,8.973214,17.197696,1.763771e+09,10.709130,56.782350,30.746397,10.641595,...,35.677911,31.484341,2143.860961,4.260847,3891.794867,1999.091524,0.0,1.485460e+05,-8.436869,1.482873e+05
std,2.214471e+07,1.607652e+06,8.584042,7.720066,11.148180,3.929193e+07,5.612241,454.775267,23.509615,6.009981,...,29.485718,31.496186,2367.679880,0.645695,4493.832586,10.117704,0.0,1.612067e+06,2158.634050,1.607653e+06
min,1.667838e+06,2.300000e+03,1.000000,-2.000000,1.000000,1.255421e+09,1.000000,6.000000,0.200000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1921.000000,0.0,2.300000e+03,-436000.000000,2.300000e+03
25%,3.263985e+08,5.500000e+04,10.000000,4.000000,9.000000,1.769941e+09,6.000000,35.000000,19.000000,7.000000,...,0.000000,0.000000,169.000000,4.200000,172.000000,1994.000000,0.0,5.500000e+04,0.000000,5.500000e+04
50%,3.269312e+08,7.300000e+04,15.000000,7.000000,16.000000,1.771231e+09,10.000000,43.000000,24.000000,9.900000,...,50.000000,50.000000,839.000000,4.400000,1451.000000,1994.000000,0.0,7.300000e+04,0.000000,7.300000e+04
75%,3.277866e+08,1.100000e+05,21.000000,12.000000,22.000000,1.773429e+09,14.000000,60.000000,35.000000,12.000000,...,50.000000,50.000000,4041.000000,4.500000,9152.000000,2005.000000,0.0,1.100000e+05,0.000000,1.100000e+05
max,3.283110e+08,1.400000e+08,50.000000,81.000000,117.000000,1.774642e+09,30.000000,80000.000000,500.000000,200.000000,...,100.000000,100.000000,6082.000000,5.000000,19821.000000,2022.000000,0.0,1.400000e+08,70000.000000,1.400000e+08


In [8]:
# Используем актуальную цену для анализа (из price_changes или offers.price)
if 'price_actual' not in df.columns:
    # Если ячейка 6 не выполнилась, используем offers.price
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
else:
    df['price'] = pd.to_numeric(df['price_actual'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Гистограмма
axes[0, 0].hist(df['price'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Распределение цен (гистограмма)')
axes[0, 0].set_xlabel('Цена (руб.)')
axes[0, 0].set_ylabel('Частота')
axes[0, 0].grid(True, alpha=0.3)

# Boxplot
axes[0, 1].boxplot(df['price'].dropna(), vert=True)
axes[0, 1].set_title('Распределение цен (boxplot)')
axes[0, 1].set_ylabel('Цена (руб.)')
axes[0, 1].grid(True, alpha=0.3)

# Логарифмированная шкала
log_price = np.log1p(df['price'].dropna())
axes[1, 0].hist(log_price, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Распределение цен (логарифмированная шкала)')
axes[1, 0].set_xlabel('log(Цена + 1)')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].grid(True, alpha=0.3)

# Q-Q plot для проверки нормальности
from scipy import stats
stats.probplot(df['price'].dropna(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot цены')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Статистика по цене:")
print(df['price'].describe())
print(f"\nМедиана: {df['price'].median():.2f}")
print(f"Среднее: {df['price'].mean():.2f}")
print(f"Мода: {df['price'].mode().values[0] if len(df['price'].mode()) > 0 else 'N/A'}")


Статистика по цене:
count    4.677900e+04
mean     1.482873e+05
std      1.607653e+06
min      2.300000e+03
25%      5.500000e+04
50%      7.300000e+04
75%      1.100000e+05
max      1.400000e+08
Name: price, dtype: float64

Медиана: 73000.00
Среднее: 148287.31
Мода: 60000.0


### 3.2 Распределение числовых признаков


In [9]:
key_numeric = ['total_area', 'living_area', 'kitchen_area', 'floor_number', 
               'floors_count', 'build_year', 'rooms_count', 'ceiling_height']
key_numeric = [col for col in key_numeric if col in df.columns]

fig, axes = plt.subplots(len(key_numeric), 2, figsize=(15, 4 * len(key_numeric)))

for i, col in enumerate(key_numeric):
    if col in df.columns:
        data = pd.to_numeric(df[col], errors='coerce').dropna()
        
        if len(data) > 0:
            # Гистограмма
            axes[i, 0].hist(data, bins=30, edgecolor='black', alpha=0.7)
            axes[i, 0].set_title(f'Распределение {col}')
            axes[i, 0].set_xlabel(col)
            axes[i, 0].set_ylabel('Частота')
            axes[i, 0].grid(True, alpha=0.3)
            
            # Boxplot
            axes[i, 1].boxplot(data, vert=True)
            axes[i, 1].set_title(f'Boxplot {col}')
            axes[i, 1].set_ylabel(col)
            axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 3.3 Распределение категориальных признаков


In [10]:
cat_cols = ['category', 'district', 'repair_type', 'material_type', 'realty_type', 
            'deal_type', 'flat_type', 'rooms_count']
cat_cols = [col for col in cat_cols if col in df.columns]

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(15, 5 * len(cat_cols)))

for i, col in enumerate(cat_cols):
    if col in df.columns:
        value_counts = df[col].value_counts().head(15)
        
        axes[i].barh(range(len(value_counts)), value_counts.values)
        axes[i].set_yticks(range(len(value_counts)))
        axes[i].set_yticklabels(value_counts.index)
        axes[i].set_title(f'{col} (топ-15)')
        axes[i].set_xlabel('Количество')
        axes[i].grid(True, alpha=0.3, axis='x')
        
        axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

# Выводим статистику по категориальным признакам
for col in cat_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Уникальных значений: {df[col].nunique()}")
        print(f"  Пропусков: {df[col].isna().sum()} ({df[col].isna().sum() / len(df) * 100:.1f}%)")
        print(f"  Топ-5:")
        print(df[col].value_counts().head())



category:
  Уникальных значений: 1
  Пропусков: 0 (0.0%)
  Топ-5:
category
flatRent    46779
Name: count, dtype: int64

district:
  Уникальных значений: 126
  Пропусков: 4772 (10.2%)
  Топ-5:
district
Пресненский            1620
Хорошево-Мневники      1187
Раменки                1119
Очаково-Матвеевское     971
Даниловский             912
Name: count, dtype: int64

repair_type:
  Уникальных значений: 4
  Пропусков: 2069 (4.4%)
  Топ-5:
repair_type
euro        18980
design      13343
cosmetic    11823
no            564
Name: count, dtype: int64

material_type:
  Уникальных значений: 8
  Пропусков: 4543 (9.7%)
  Топ-5:
material_type
panel       11425
none        11406
monolith     9197
brick        6697
block        2769
Name: count, dtype: int64

realty_type:
  Уникальных значений: 1
  Пропусков: 0 (0.0%)
  Топ-5:


realty_type
flat    46779
Name: count, dtype: int64

deal_type:
  Уникальных значений: 1
  Пропусков: 0 (0.0%)
  Топ-5:
deal_type
rent    46779
Name: count, dtype: int64

flat_type:
  Уникальных значений: 3
  Пропусков: 0 (0.0%)
  Топ-5:
flat_type
rooms       41248
studio       5412
openPlan      119
Name: count, dtype: int64

rooms_count:
  Уникальных значений: 6
  Пропусков: 5531 (11.8%)
  Топ-5:
rooms_count
1.0    17431
2.0    15333
3.0     6330
4.0     1436
5.0      488
Name: count, dtype: int64


## 4. Анализ пропусков


In [11]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Пропусков': missing,
    'Процент': missing_percent
}).sort_values('Процент', ascending=False)

print("Колонки с пропусками:")
print(missing_df[missing_df['Пропусков'] > 0])

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
cols_with_missing = missing_df[missing_df['Пропусков'] > 0].index[:30]
if len(cols_with_missing) > 0:
    sns.heatmap(df[cols_with_missing].isnull(), yticklabels=False, 
                cbar=True, cmap='viridis', ax=axes[0])
    axes[0].set_title('Heatmap пропусков (топ-30 колонок)')
    
    top_missing = missing_df.head(20)
    axes[1].barh(range(len(top_missing)), top_missing['Процент'].values)
    axes[1].set_yticks(range(len(top_missing)))
    axes[1].set_yticklabels(top_missing.index, fontsize=8)
    axes[1].set_xlabel('Процент пропусков')
    axes[1].set_title('Топ-20 колонок по проценту пропусков')
    axes[1].grid(True, alpha=0.3, axis='x')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


Колонки с пропусками:
                           Пропусков     Процент
is_mortgage_allowed            46779  100.000000
views_count                    46779  100.000000
is_penthouse                   42730   91.344407
developer_foundation_year      27549   58.891810
developer_review_count         27547   58.887535
developer_rate                 27547   58.887535
developer_name                 27299   58.357383
developer_buildings_count      27299   58.357383
developer_is_reliable          27089   57.908463
garbage_chute                  22842   48.829603
project_type                   17758   37.961478
windows_view                   16100   34.417153
ceiling_height                 15089   32.255927
cargo_lifts                    12598   26.930888
entrances                      12441   26.595267
living_area                    11128   23.788452
parking_type                   10837   23.166378
kitchen_area                    7986   17.071763
lifts_count                     7555   16.15040

### Как заполнять будем пропуски

- **Числовые признаки**: медиана или среднее, в зависимости от распределения
- **Категориальные признаки**: мода или "unknown"
- **Координаты**: могут быть критичными для geo-фичей
- **build_year**: можно попробовать восстановить по району/материалу


## 5. Выбросы

### 5.1 IQR метод


In [12]:
def detect_outliers_iqr(df, column):
    """Обнаружение выбросов методом IQR"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

key_columns = ['price', 'total_area', 'living_area', 'floor_number', 'build_year']
outliers_summary = {}
for col in key_columns:
    if col in df.columns:
        df_col = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(df_col) > 0:
            outliers, lower, upper = detect_outliers_iqr(df, col)
            outliers_summary[col] = {
                'count': len(outliers),
                'percent': len(outliers) / len(df) * 100,
                'lower_bound': lower,
                'upper_bound': upper,
                'min_value': df[col].min(),
                'max_value': df[col].max()
            }

print("Выбросы (IQR метод):")
for col, stats in outliers_summary.items():
    print(f"\n{col}:")
    print(f"  Выбросов: {stats['count']} ({stats['percent']:.1f}%)")
    print(f"  Границы: [{stats['lower_bound']:.2f}, {stats['upper_bound']:.2f}]")
    print(f"  Диапазон данных: [{stats['min_value']:.2f}, {stats['max_value']:.2f}]")

fig, axes = plt.subplots(len(outliers_summary), 1, figsize=(12, 4 * len(outliers_summary)))

for i, (col, stats) in enumerate(outliers_summary.items()):
    data = pd.to_numeric(df[col], errors='coerce').dropna()
    axes[i].boxplot(data, vert=True)
    axes[i].axhline(y=stats['lower_bound'], color='r', linestyle='--', alpha=0.5, label='Lower bound')
    axes[i].axhline(y=stats['upper_bound'], color='r', linestyle='--', alpha=0.5, label='Upper bound')
    axes[i].set_title(f'Выбросы в {col} (IQR метод)')
    axes[i].set_ylabel(col)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Выбросы (IQR метод):

price:
  Выбросов: 4984 (10.7%)
  Границы: [-27500.00, 192500.00]
  Диапазон данных: [2300.00, 140000000.00]

total_area:
  Выбросов: 3564 (7.6%)
  Границы: [-2.50, 97.50]
  Диапазон данных: [6.00, 80000.00]

living_area:
  Выбросов: 2664 (5.7%)
  Границы: [-5.00, 59.00]
  Диапазон данных: [0.20, 500.00]

floor_number:
  Выбросов: 2029 (4.3%)
  Границы: [-8.00, 24.00]
  Диапазон данных: [-2.00, 81.00]

build_year:
  Выбросов: 75 (0.2%)
  Границы: [1896.00, 2096.00]
  Диапазон данных: [1785.00, 2027.00]


### 5.2 Z-score метод


In [13]:
def detect_outliers_zscore(df, column, threshold=3):
    """Обнаружение выбросов методом Z-score"""
    df_col = pd.to_numeric(df[column], errors='coerce')
    z_scores = np.abs((df_col - df_col.mean()) / df_col.std())
    outliers = df[z_scores > threshold]
    return outliers, z_scores

zscore_summary = {}
for col in key_columns:
    if col in df.columns:
        df_col = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(df_col) > 0 and df_col.std() > 0:
            outliers, z_scores = detect_outliers_zscore(df, col, threshold=3)
            zscore_summary[col] = {
                'count': len(outliers),
                'percent': len(outliers) / len(df) * 100,
                'max_z_score': z_scores.max() if len(z_scores) > 0 else 0
            }

print("Выбросы (Z-score метод, threshold=3):")
for col, stats in zscore_summary.items():
    print(f"{col}: {stats['count']} выбросов ({stats['percent']:.1f}%), max Z-score: {stats['max_z_score']:.2f}")


Выбросы (Z-score метод, threshold=3):
price: 29 выбросов (0.1%), max Z-score: 86.99
total_area: 3 выбросов (0.0%), max Z-score: 175.79
living_area: 635 выбросов (1.4%), max Z-score: 19.96
floor_number: 717 выбросов (1.5%), max Z-score: 9.33
build_year: 406 выбросов (0.9%), max Z-score: 7.54


### 5.3 Анализ экстремальных значений цены

Проверим самые дорогие и дешевые предложения


In [14]:
print("Топ-10 самых дорогих:")
print(df.nlargest(10, 'price')[['cian_id', 'price', 'total_area', 'district', 'rooms_count', 'repair_type']].to_string())

print("\nТоп-10 самых дешевых:")
print(df.nsmallest(10, 'price')[['cian_id', 'price', 'total_area', 'district', 'rooms_count', 'repair_type']].to_string())

df['price_per_sqm'] = df['price'] / pd.to_numeric(df['total_area'], errors='coerce')

print("\nСтатистика цены за м²:")
print(df['price_per_sqm'].describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(df['price_per_sqm'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Распределение цены за м²')
axes[0].set_xlabel('Цена за м² (руб.)')
axes[0].set_ylabel('Частота')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df['price_per_sqm'].dropna(), vert=True)
axes[1].set_title('Boxplot цены за м²')
axes[1].set_ylabel('Цена за м² (руб.)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Топ-10 самых дорогих:


         cian_id        price  total_area         district  rooms_count repair_type
29082  327250011  140000000.0        60.0         Якиманка          3.0        euro
17389  326667118   99000000.0        41.0      Даниловский          1.0        euro
38531  327946950   90000000.0        59.2      Теплый Стан          3.0    cosmetic
23488  326993176   87000000.0        45.0  Чертаново Южное          2.0        euro
16229  326627380   85000000.0        72.0         Царицыно          3.0        euro
22375  326917938   85000000.0        60.0        Таганский          2.0    cosmetic
30854  327379604   80000000.0        45.0      Теплый Стан          2.0        euro
39278  327969523   80000000.0        58.0      Даниловский          2.0    cosmetic
2148   325733126   75000000.0        32.0        Можайский          NaN      design
17022  326640739   75000000.0        67.0             None          3.0        euro

Топ-10 самых дешевых:


         cian_id    price  total_area              district  rooms_count repair_type
112    320322977   2300.0        56.4          Дорогомилово          2.0      design
597    306974586   5900.0       297.0             Басманный          6.0      design
42454  328092324   7000.0       189.0              Тверской          4.0        euro
478    315558243   7500.0       189.0              Тверской          NaN      design
603    307147680   8000.0       412.0         Ломоносовский          6.0        euro
832    291334315   8000.0       191.0  Проспект Вернадского          5.0      design
2566   325756948  10000.0        20.0                  None          NaN          no
594    323644992  12000.0       380.0                 Арбат          6.0      design
33862  327605385  14000.0        10.0              Тверской          NaN          no
1211   309431278  15000.0       380.1   Очаково-Матвеевское          5.0      design

Статистика цены за м²:
count    4.677900e+04
mean     2.676952e+

## 6. Временная валидация (train / test)

**Зачем менять логику:** разбиение «последние 30 дней от максимума» по **`publication_at`** при длинном хвосте старых публикаций даёт **перекос**: почти все наблюдения с недавней датой публикации оказываются в test, train — узким срезом.

**Рекомендация для ML и отчёта:** основной порог для time-split — **`offer_created_at`** (или согласованно `publication_at`, если задача явно про «дату на рынке»). Test: **последние 14 дней** или **последний календарный месяц**; train — всё раньше. Ниже — рекомендуемое разбиение по `created_at` и сравнение с «30 дней от max» по `publication_at`.


In [15]:
df['offer_created_at'] = pd.to_datetime(df['offer_created_at'], errors='coerce')
df_ca = df.dropna(subset=['offer_created_at']).copy()
max_ca = df_ca['offer_created_at'].max()
split_ca = max_ca - pd.Timedelta(days=14)
train_ca = df_ca[df_ca['offer_created_at'] < split_ca]
test_ca = df_ca[df_ca['offer_created_at'] >= split_ca]

print("=== Разбиение по offer_created_at (рекомендуемый сценарий: test = последние 14 дней) ===")
print(f"Диапазон created_at: {df_ca['offer_created_at'].min()} — {max_ca}")
print(f"Порог (train < / test >=): {split_ca}")
print(f"Train: {len(train_ca)} ({len(train_ca)/len(df_ca)*100:.1f}%), Test: {len(test_ca)} ({len(test_ca)/len(df_ca)*100:.1f}%)")
if len(train_ca) > 0 and len(test_ca) > 0:
    ok = train_ca['offer_created_at'].max() < test_ca['offer_created_at'].min()
    print(f"Все train строго раньше test: {ok}")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
daily_ca = df_ca.groupby(df_ca['offer_created_at'].dt.floor('D')).size()
daily_ca.plot(ax=axes[0], title='Объём по дням (offer_created_at)', color='steelblue')
axes[0].axvline(split_ca, color='green', linestyle='--', label='Порог train|test')
axes[0].legend()
axes[0].set_ylabel('Кол-во')

axes[1].scatter(train_ca['offer_created_at'], train_ca['price'], alpha=0.25, s=8, label='Train')
axes[1].scatter(test_ca['offer_created_at'], test_ca['price'], alpha=0.35, s=10, color='red', label='Test')
axes[1].axvline(split_ca, color='green', linestyle='--')
axes[1].set_title('Цена vs offer_created_at')
axes[1].legend()
plt.tight_layout()
plt.show()

df['publication_date'] = pd.to_datetime(df['publication_at'], unit='s', errors='coerce')
df_pub = df.dropna(subset=['publication_date']).copy()
split_pub = df_pub['publication_date'].max() - pd.Timedelta(days=30)
tr_p = df_pub[df_pub['publication_date'] <= split_pub]
te_p = df_pub[df_pub['publication_date'] > split_pub]

print("\n=== Сравнение: publication_at, последние 30 дней от max (может давать перекос) ===")
print(f"Диапазон publication: {df_pub['publication_date'].min()} — {df_pub['publication_date'].max()}")
print(f"Train: {len(tr_p)} ({len(tr_p)/len(df_pub)*100:.1f}%), Test: {len(te_p)} ({len(te_p)/len(df_pub)*100:.1f}%)")
print("При смеси очень старых и новых публикаций эта схема часто не подходит для оценки качества; предпочтительнее split по created_at.")


=== Разбиение по offer_created_at (рекомендуемый сценарий: test = последние 14 дней) ===
Диапазон created_at: 2026-01-09 15:35:40 — 2026-03-27 23:32:21
Порог (train < / test >=): 2026-03-13 23:32:21
Train: 34515 (73.8%), Test: 12264 (26.2%)
Все train строго раньше test: True



=== Сравнение: publication_at, последние 30 дней от max (может давать перекос) ===
Диапазон publication: 2009-10-13 07:59:21 — 2026-03-27 20:06:49
Train: 29511 (63.1%), Test: 17256 (36.9%)
При смеси очень старых и новых публикаций эта схема часто не подходит для оценки качества; предпочтительнее split по created_at.


## 7. Корреляции

Анализ взаимосвязей между признаками


In [16]:
numeric_for_corr = ['price', 'total_area', 'living_area', 'kitchen_area', 
                    'floor_number', 'floors_count', 'build_year', 'rooms_count',
                    'ceiling_height', 'views_count', 'photos_count']
numeric_for_corr = [col for col in numeric_for_corr if col in df.columns]

corr_matrix = df[numeric_for_corr].apply(pd.to_numeric, errors='coerce').corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()

if 'price' in corr_matrix.columns:
    price_corr = corr_matrix['price'].sort_values(ascending=False)
    print("\nКорреляции с ценой:")
    print(price_corr)



Корреляции с ценой:
price             1.000000
living_area       0.071174
rooms_count       0.065254
kitchen_area      0.036834
photos_count      0.028008
total_area        0.007534
floor_number      0.006149
ceiling_height    0.001147
floors_count      0.000997
build_year       -0.006249
views_count            NaN
Name: price, dtype: float64


## 8. Краткий анализ результатов EDA

### Основные находки по каждой ячейке:


In [17]:
# Итоговый анализ результатов
print("=" * 50)
print("АНАЛИЗ РЕЗУЛЬТАТОВ EDA")
print("=" * 50)

print(f"\n1. Размер датасета:")
print(f"   Записей: {len(df)}, колонок: {len(df.columns)}")

if 'category' in df.columns:
    print(f"\n   Фильтрация category:")
    category_counts = df['category'].value_counts()
    print(f"   {category_counts.to_dict()}")
    if 'dailyFlatRent' in category_counts.index:
        print(f"   dailyFlatRent: {category_counts['dailyFlatRent']} записей (отфильтровано)")

print(f"\n2. Цены:")
if 'price_actual' in df.columns:
    price_col = 'price_actual'
else:
    price_col = 'price'
    
price_stats = df[price_col].describe()
print(f"   Медиана: {price_stats['50%']:,.0f} руб")
print(f"   Среднее: {price_stats['mean']:,.0f} руб")
print(f"   Min: {price_stats['min']:,.0f} руб")
print(f"   Max: {price_stats['max']:,.0f} руб")
print(f"   Max цена {price_stats['max']:,.0f} руб - выброс")
print(f"   Std: {price_stats['std']:,.0f} руб")

if 'price_changes' in df.columns:
    price_changes_count = df['price_changes'].notna().sum()
    price_matches_count = df['price_matches'].sum() if 'price_matches' in df.columns else 0
    print(f"\n   Анализ price_changes:")
    print(f"   Записей с price_changes: {price_changes_count} ({price_changes_count/len(df)*100:.1f}%)")
    print(f"   Цены совпадают: {price_matches_count} ({price_matches_count/len(df)*100:.1f}%)")
    if 'price_actual' in df.columns:
        from_changes = (df['price_actual'] == df['price_from_changes']).sum() if 'price_from_changes' in df.columns else 0
        from_offers = (df['price_actual'] == df['price']).sum() if 'price' in df.columns else 0
        print(f"   Используется: price_changes={from_changes}, offers.price={from_offers}")

if 'price_per_sqm' in df.columns:
    price_sqm = df['price_per_sqm'].dropna()
    if len(price_sqm) > 0:
        print(f"\n   Цена за м²:")
        print(f"   Медиана: {price_sqm.median():,.0f} руб/м²")
        print(f"   Mean: {price_sqm.mean():,.0f} руб/м²")
        print(f"   Max: {price_sqm.max():,.0f} руб/м² (выброс)")

# Проверка выбросов
Q1 = price_stats['25%']
Q3 = price_stats['75%']
IQR = Q3 - Q1
outliers_count = len(df[(df[price_col] < Q1 - 1.5*IQR) | (df[price_col] > Q3 + 1.5*IQR)])
print(f"   Выбросов (IQR): {outliers_count} ({outliers_count/len(df)*100:.1f}%)")

print(f"\n3. Пропуски (топ-10):")
missing_top10 = df.isnull().sum().sort_values(ascending=False).head(10)
for col, count in missing_top10.items():
    if col not in ['price_diff', 'price_from_changes', 'price_matches', 'price_changes']:
        print(f"   {col}: {count} ({count/len(df)*100:.1f}%)")

print(f"\n4. Корреляции с ценой:")
if 'total_area' in df.columns and price_col in df.columns:
    corr_total = df[['total_area', price_col]].apply(pd.to_numeric, errors='coerce').corr().iloc[0, 1]
    print(f"   total_area: {corr_total:.4f}")
if 'kitchen_area' in df.columns and price_col in df.columns:
    corr_kitchen = df[['kitchen_area', price_col]].apply(pd.to_numeric, errors='coerce').corr().iloc[0, 1]
    print(f"   kitchen_area: {corr_kitchen:.4f}")

print(f"\n5. Категориальные признаки:")
if 'category' in df.columns:
    print(f"   category:")
    print(f"      {df['category'].value_counts().to_dict()}")

print(f"   deal_type:")
if 'deal_type' in df.columns:
    print(f"      {df['deal_type'].value_counts().to_dict()}")

print(f"\n6. Временное разделение (рекомендуемый учёт для отчёта):")
oca = pd.to_datetime(df['offer_created_at'], errors='coerce').dropna()
if len(oca):
    mx = oca.max()
    sp = mx - pd.Timedelta(days=14)
    tr = (oca < sp).sum()
    te = (oca >= sp).sum()
    print(f"   created_at: {oca.min()} — {mx}")
    print(f"   Train/Test (test = последние 14 дней по created_at): {tr}/{te} ({tr/len(oca)*100:.1f}%/{te/len(oca)*100:.1f}%)")
if 'publication_date' in df.columns:
    df_with_date = df.dropna(subset=['publication_date'])
    split_date = df_with_date['publication_date'].max() - pd.Timedelta(days=30)
    train_size = len(df_with_date[df_with_date['publication_date'] <= split_date])
    test_size = len(df_with_date[df_with_date['publication_date'] > split_date])
    print(f"   Сравнение — publication_at, последние 30 дней от max: {train_size}/{test_size} ({train_size/len(df_with_date)*100:.1f}%/{test_size/len(df_with_date)*100:.1f}%) (может быть перекошено)")

print(f"\n7. Критические проблемы:")
issues = []
if price_stats['max'] > 100000000:
    issues.append(f"Выбросы в цене: max={price_stats['max']:,.0f} руб")
if 'deal_type' in df.columns and (df['deal_type'] == 'sale').any():
    issues.append(f"deal_type='sale': {len(df[df['deal_type'] == 'sale'])} шт (при SQL-фильтре rent не должно быть)")
if 'total_area' in df.columns and df['total_area'].isna().sum() / len(df) > 0.1:
    issues.append(f"Пропуски в total_area: {df['total_area'].isna().sum()/len(df)*100:.1f}%")
if 'price_changes' in df.columns and 'price_matches' in df.columns:
    diff_count = (~df['price_matches']).sum()
    if diff_count > 0:
        issues.append(f"Различия price_changes и offers.price: {diff_count} записей (0.2%)")
if 'category' in df.columns and (df['category'] == 'dailyFlatRent').any():
    daily_count = len(df[df['category'] == 'dailyFlatRent'])
    issues.append(f"dailyFlatRent в данных: {daily_count} записей (при SQL flatRent не должно быть)")

if issues:
    for i, issue in enumerate(issues, 1):
        print(f"   {i}. {issue}")
else:
    print("   Не найдено критических проблем")

print("\n" + "=" * 50)


АНАЛИЗ РЕЗУЛЬТАТОВ EDA

1. Размер датасета:
   Записей: 46779, колонок: 69

   Фильтрация category:
   {'flatRent': 46779}

2. Цены:
   Медиана: 73,000 руб
   Среднее: 148,287 руб
   Min: 2,300 руб
   Max: 140,000,000 руб
   Max цена 140,000,000 руб - выброс
   Std: 1,607,653 руб

   Анализ price_changes:
   Записей с price_changes: 46522 (99.5%)
   Цены совпадают: 46693 (99.8%)
   Используется: price_changes=46522, offers.price=46779

   Цена за м²:
   Медиана: 1,750 руб/м²
   Mean: 2,677 руб/м²
   Max: 2,414,634 руб/м² (выброс)
   Выбросов (IQR): 4984 (10.7%)

3. Пропуски (топ-10):
   is_mortgage_allowed: 46779 (100.0%)
   views_count: 46779 (100.0%)
   is_penthouse: 42730 (91.3%)
   developer_foundation_year: 27549 (58.9%)
   developer_review_count: 27547 (58.9%)
   developer_rate: 27547 (58.9%)
   developer_buildings_count: 27299 (58.4%)
   developer_name: 27299 (58.4%)
   developer_is_reliable: 27089 (57.9%)
   garbage_chute: 22842 (48.8%)

4. Корреляции с ценой:
   total_area: 0.

   Сравнение — publication_at, последние 30 дней от max: 29511/17256 (63.1%/36.9%) (может быть перекошено)

7. Критические проблемы:
   1. Выбросы в цене: max=140,000,000 руб
   2. Различия price_changes и offers.price: 86 записей (0.2%)



## 9. Анализ дубликатов

Проверка дубликатов объявлений по этаж + адрес + площадь + комнаты


In [18]:
df['floor_number'] = pd.to_numeric(df['floor_number'], errors='coerce')
df['total_area'] = pd.to_numeric(df['total_area'], errors='coerce')
df['rooms_count'] = pd.to_numeric(df['rooms_count'], errors='coerce')

df['street'] = df['street'].fillna('').astype(str).str.strip().str.lower()
df['house'] = df['house'].fillna('').astype(str).str.strip().str.lower()

df['address_key'] = df['street'].fillna('') + '_' + df['house'].fillna('')
df['area_rounded'] = df['total_area'].round(0)
df['rooms_rounded'] = df['rooms_count'].round(0)

df['duplicate_key'] = (
    df['address_key'].astype(str) + '_' +
    df['floor_number'].astype(str) + '_' +
    df['area_rounded'].astype(str) + '_' +
    df['rooms_rounded'].astype(str)
)

duplicates = df[df.duplicated(subset=['duplicate_key'], keep=False)].sort_values('duplicate_key')

print(f"Всего записей: {len(df)}")
print(f"Уникальных ключей: {df['duplicate_key'].nunique()}")
print(f"Дубликатов: {len(df) - df['duplicate_key'].nunique()}")

if len(duplicates) > 0:
    print(f"\nПримеры дубликатов (первые 10):")
    print(duplicates[['cian_id', 'address_key', 'floor_number', 'total_area', 'rooms_count', 'price_actual', 'publication_at']].head(10).to_string())
    
    df['publication_date'] = pd.to_datetime(df['publication_at'], unit='s', errors='coerce')
    duplicates_with_date = duplicates.dropna(subset=['publication_date'])
    if len(duplicates_with_date) > 0:
        print(f"\nСтатистика дубликатов:")
        print(f"  Групп дубликатов: {duplicates_with_date.groupby('duplicate_key').size().shape[0]}")
        print(f"  Средний размер группы: {duplicates_with_date.groupby('duplicate_key').size().mean():.1f}")
        print(f"  Максимальный размер группы: {duplicates_with_date.groupby('duplicate_key').size().max()}")
else:
    print("Дубликатов не найдено")


Всего записей: 46779
Уникальных ключей: 41591
Дубликатов: 5188

Примеры дубликатов (первые 10):
         cian_id                      address_key  floor_number  total_area  rooms_count  price_actual  publication_at
8219   326296445     1-й балтийский переулок_3/25             6        80.0          3.0      140000.0    1.769621e+09
7902   326289341     1-й балтийский переулок_3/25             6        80.0          3.0      140000.0    1.769611e+09
16297  326626199        1-й войковский проезд_4к1             3        45.0          2.0       70000.0    1.770546e+09
28708  327219230        1-й войковский проезд_4к1             3        45.0          2.0       70000.0    1.772010e+09
33096  327584164        1-й войковский проезд_4к1             3        45.0          2.0       65000.0    1.772870e+09
11510  326424704        1-й войковский проезд_4к1             3        45.0          2.0       65000.0    1.770021e+09
6527   326240475  1-й грайвороновский проезд_13к1            23        

## 10. Выводы и рекомендации


In [19]:
print("=" * 60)
print("ВЫВОДЫ EDA")
print("=" * 60)

print(f"\n1. Размер датасета: {len(df)} записей, {len(df.columns)} колонок")

if 'price_actual' in df.columns:
    price_col = 'price_actual'
else:
    price_col = 'price'

price_stats = df[price_col].describe()
print(f"\n2. Цены:")
print(f"   Медиана: {price_stats['50%']:,.0f} руб")
print(f"   Среднее: {price_stats['mean']:,.0f} руб")
print(f"   Min/Max: {price_stats['min']:,.0f} / {price_stats['max']:,.0f} руб")

Q1 = price_stats['25%']
Q3 = price_stats['75%']
IQR = Q3 - Q1
outliers_count = len(df[(df[price_col] < Q1 - 1.5*IQR) | (df[price_col] > Q3 + 1.5*IQR)])
print(f"   Выбросов (IQR): {outliers_count} ({outliers_count/len(df)*100:.1f}%)")

if 'price_changes' in df.columns:
    price_changes_count = df['price_changes'].notna().sum()
    print(f"\n3. price_changes: {price_changes_count} записей ({price_changes_count/len(df)*100:.1f}%)")

if 'duplicate_key' in df.columns:
    duplicates_count = len(df) - df['duplicate_key'].nunique()
    print(f"\n4. Дубликаты: {duplicates_count} записей ({duplicates_count/len(df)*100:.1f}%)")

print(f"\n5. Пропуски (топ-5):")
missing_top5 = df.isnull().sum().sort_values(ascending=False).head(5)
for col, count in missing_top5.items():
    if count > 0:
        print(f"   {col}: {count} ({count/len(df)*100:.1f}%)")

if 'total_area' in df.columns and price_col in df.columns:
    corr_total = df[['total_area', price_col]].apply(pd.to_numeric, errors='coerce').corr().iloc[0, 1]
    print(f"\n6. Корреляции с ценой:")
    print(f"   total_area: {corr_total:.4f}")
    if 'kitchen_area' in df.columns:
        corr_kitchen = df[['kitchen_area', price_col]].apply(pd.to_numeric, errors='coerce').corr().iloc[0, 1]
        print(f"   kitchen_area: {corr_kitchen:.4f}")

print(f"\n7. Рекомендации:")
print("   - В выборке: category=flatRent и deal_type=rent (уже в SQL)")
print("   - Валидация: time-based split по offer_created_at (test — последние 14 дней или месяц), не опираться на «30 дней от max» по publication_at при длинном хвосте публикаций")
print("   - Удалить/каппинг выбросов по цене и площади; дубликаты — оставить более позднее по publication/created")
print("   - Целевая цена: price_actual из price_changes")
print("   - Линейные корреляции с ценой слабые — geo, цена за м², нелинейные модели")
print("=" * 60)


ВЫВОДЫ EDA

1. Размер датасета: 46779 записей, 73 колонок

2. Цены:
   Медиана: 73,000 руб
   Среднее: 148,287 руб
   Min/Max: 2,300 / 140,000,000 руб
   Выбросов (IQR): 4984 (10.7%)

3. price_changes: 46522 записей (99.5%)

4. Дубликаты: 5188 записей (11.1%)

5. Пропуски (топ-5):


   views_count: 46779 (100.0%)
   is_mortgage_allowed: 46779 (100.0%)
   is_penthouse: 42730 (91.3%)
   developer_foundation_year: 27549 (58.9%)
   developer_review_count: 27547 (58.9%)

6. Корреляции с ценой:
   total_area: 0.0075
   kitchen_area: 0.0368

7. Рекомендации:
   - В выборке: category=flatRent и deal_type=rent (уже в SQL)
   - Валидация: time-based split по offer_created_at (test — последние 14 дней или месяц), не опираться на «30 дней от max» по publication_at при длинном хвосте публикаций
   - Удалить/каппинг выбросов по цене и площади; дубликаты — оставить более позднее по publication/created
   - Целевая цена: price_actual из price_changes
   - Линейные корреляции с ценой слабые — geo, цена за м², нелинейные модели
